In [ ]:
# Fine-Tuning LLMs with Hugging Face

## Step 1: Installing and importing the libraries

!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 trl==0.4.7 gradio streamlit
!pip install huggingface_hub

import torch

from trl import SFTTrainer
from peft import LoraConfig
from datasets import load_dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, pipeline)


In [ ]:
# Step 2: Loading the model

llama_model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path = "aboonaji/llama2finetune-v2",

                                                   quantization_config = BitsAndBytesConfig(load_in_4bit = True, bnb_4bit_compute_dtype = getattr(torch, "float16"), bnb_4bit_quant_type = "nf4"))

llama_model.config.use_cache = False

llama_model.config.pretraining_tp = 1


In [ ]:
# Step 3: Loading the tokenizer

llama_tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path = "aboonaji/llama2finetune-v2", trust_remote_code = True)

llama_tokenizer.pad_token = llama_tokenizer.eos_token

llama_tokenizer.padding_side = "right"


In [ ]:
# Step 4: Setting the training arguments

training_arguments = TrainingArguments(output_dir = "./results", per_device_train_batch_size = 4, max_steps = 100)


In [ ]:
# Step 5: Creating the Supervised Fine-Tuning trainer

llama_sft_trainer = SFTTrainer(model = llama_model,

                               args = training_arguments,

                               train_dataset = load_dataset(path = "aboonaji/wiki_medical_terms_llam2_format", split = "train"),

                               tokenizer = llama_tokenizer,

                               peft_config = LoraConfig(task_type = "CAUSAL_LM", r = 64, lora_alpha = 16, lora_dropout = 0.1),

                               dataset_text_field = "text")


In [ ]:
# Step 6: Training the model

llama_sft_trainer.train()


In [ ]:
# Step 7: Chatting with the model

---

user_prompt = "Please tell me about Bursitis"

text_generation_pipeline = pipeline(task = "text-generation", model = llama_model, tokenizer = llama_tokenizer, max_length = 300)

model_answer = text_generation_pipeline(f"<s>[INST] {user_prompt} [/INST]")

print(model_answer[0]['generated_text'])

user_prompt = "Please tell me about Fever"

text_generation_pipeline = pipeline(task = "text-generation", model = llama_model, tokenizer = llama_tokenizer, max_length = 350)

model_answer = text_generation_pipeline(f"<s>[INST] {user_prompt} [/INST]")

print(model_answer[0]['generated_text'])


In [ ]:
# Step 8: Saving the model weights

llama_sft_trainer.model.save_pretrained("/content/drive/MyDrive/llama2_simplifying_medicine/llama2_simplifying_health_4bit")


In [ ]:
# Step 9: Loading the model weights

from peft import PeftModel

new_model="/content/drive/MyDrive/llama2_simplifying_medicine/llama2_simplifying_health_4bit"

model_name = "aboonaji/llama2finetune-v2"

device_map = {"": 0}

import locale

locale.getpreferredencoding = lambda: "UTF-8"

!huggingface-cli login

llama_model.push_to_hub("llama2_simplifying_health_4bit", check_pr=True)

llama_tokenizer.push_to_hub("llama2_simplifying_health_4bit",check_pr=True)


In [ ]:
# UI using Streamlit

import streamlit as st

from streamlit_chat import message as st_message

if "history" not in st.session_state:
    st.session_state.history = []

st.title("Simplifying Medicine")

def generate_answer():
    tokenizer, model = llama_tokenizer, llama_model

    user_message = st.session_state.input_text

    inputs = tokenizer(st.session_state.input_text, return_tensors="pt")

    result = model.generate(**inputs)

    message_bot = tokenizer.decode(
        result[0], skip_special_tokens=True
    )

    st.session_state.history.append({"message": user_message, "is_user": True})

    st.session_state.history.append({"message": message_bot, "is_user": False})

st.text_input("Ask something", key="input_text", on_change=generate_answer)

for i, chat in enumerate(st.session_state.history):
    st_message(**chat, key=str(i)) #unpacking


In [ ]:
# UI using Gradio
SYSTEM_PROMPT = """<s>[INST] <<SYS>>

You are a helpful bot. Your answers are clear and concise.

<</SYS>>


"""

# Formatting function for message and history
def format_message(message: str, history: list, memory_limit: int = 3) -> str:
    # always keep len(history) <= memory_limit
    if len(history) > memory_limit:
        history = history[-memory_limit:]

    if len(history) == 0:
        return SYSTEM_PROMPT + f"{message} [/INST]"

    formatted_message = SYSTEM_PROMPT + f"{history[0][0]} [/INST] {history[0][1]} </s>"

    # Handle conversation history
    for user_msg, model_answer in history[1:]:
        formatted_message += f"<s>[INST] {user_msg} [/INST] {model_answer} </s>"

    # Handle the current message
    formatted_message += f"<s>[INST] {message} [/INST]"

    return formatted_message

# Generate a response from the Llama model
def get_llama_response(message: str, history: list) -> str:
    query = format_message(message, history)
    sequences = pipeline(task = "text-generation", model = llama_model, tokenizer = llama_tokenizer, max_length = 150)

    model_answer = sequences(query)

    generated_text = model_answer[0]['generated_text']
    response = generated_text[len(query):]  # Remove the prompt from the output

    print("Chatbot:", response.strip())
    return response.strip()


In [ ]:
!pip install gradio

import gradio as gr

gr.ChatInterface(get_llama_response).launch()
